# 11 — BERTopic Checkpoints: Distance Bands & Year Slices

Fits **10 independent BERTopic models** and saves each as a `.pkl` checkpoint:

| Run ID | Slice |
| --- | --- |
| `coast_band_A` | `distance2coastline < 0.1 km` (beachfront) |
| `coast_band_B` | `0.1 ≤ distance2coastline < 1.0 km` (near-coast) |
| `coast_band_C` | `distance2coastline ≥ 1.0 km` (inland) |
| `year_2018` … `year_2024` | one model per review year |

Results (topic assignments + top-word tables) are written to `TOPIC_LABELS` and `REVIEW_TOPICS` in `hotel_reviews.db`.
LLM labelling is done separately in the next notebook.

**Prerequisites**
```bash
uv run python src/preprocess_to_duckdb.py   # REVIEW_TEXT_PROCESSED
uv run python src/embed_to_duckdb.py        # REVIEW_EMBEDDINGS
```

## Section 1 — Imports & Config

In [ ]:
import sys
sys.path.insert(0, "..")

import pickle
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from stopwordsiso import stopwords
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, PartOfSpeech
from bertopic.vectorizers import ClassTfidfTransformer

from src.topic_modeling import load_from_duckdb

# ── Paths ─────────────────────────────────────────────────────────────────
DB_PATH  = Path("../data/hotel_reviews.db")
CKPT_DIR = Path("../checkpoints")
CSV_DIR  = Path("../data/topic_results")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Languages — each slice is run independently per language ───────────────
LANGUAGES = ["en", "vi"]

# ── Distance-band slices ──────────────────────────────────────────────────
# A = beachfront, B = near-coast, C = inland.
# Updated 2026-06: B widened to 0.1–1.0 km, C is now ≥ 1.0 km (was 0.1–0.5 / ≥0.5).
BAND_SLICES = {
    "coast_band_A": "r.distance2coastline < 0.1",
    "coast_band_B": "r.distance2coastline >= 0.1 AND r.distance2coastline < 1.0",
    "coast_band_C": "r.distance2coastline >= 1.0",
}

# ── Year slices ───────────────────────────────────────────────────────────
YEAR_SLICES = {f"year_{y}": y for y in range(2018, 2025)}

# ── BERTopic hyperparams ──────────────────────────────────────────────────
DEFAULT_MIN_CLUSTER = 50
SMALL_MIN_CLUSTER   = 20   # slices < 10k rows
SMALL_THRESHOLD     = 10_000

# ── Encoder — loaded once, shared across all runs ─────────────────────────
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

print(f"DB          : {DB_PATH.resolve()}")
print(f"Checkpoints : {CKPT_DIR.resolve()}")
print(f"Languages   : {LANGUAGES}")
print(f"Runs planned: {(len(BAND_SLICES) + len(YEAR_SLICES)) * len(LANGUAGES)}")

## Section 2 — Create DuckDB Tables

`TOPIC_LABELS` — one row per (run_id, topic_id): stores top words and (later) LLM seed-topic label.  
`REVIEW_TOPICS` — one row per (run_id, review_id): stores topic assignment and probability.

In [4]:
con = duckdb.connect(str(DB_PATH))

con.execute("""
    CREATE TABLE IF NOT EXISTS TOPIC_LABELS (
        run_id     VARCHAR NOT NULL,
        topic_id   INTEGER NOT NULL,
        top_words  VARCHAR,
        n_docs     INTEGER,
        seed_topic VARCHAR,
        seed_score FLOAT,
        PRIMARY KEY (run_id, topic_id)
    )
""")

con.execute("""
    CREATE TABLE IF NOT EXISTS REVIEW_TOPICS (
        run_id    VARCHAR  NOT NULL,
        review_id VARCHAR  NOT NULL,
        topic_id  INTEGER  NOT NULL,
        prob      FLOAT,
        PRIMARY KEY (run_id, review_id)
    )
""")

con.close()
print("Tables ready: TOPIC_LABELS, REVIEW_TOPICS")

Tables ready: TOPIC_LABELS, REVIEW_TOPICS


## Section 3 — Helper: Write BERTopic Results to DuckDB

In [ ]:
def fit_and_save(run_id: str, docs: list, embeddings, min_cs: int, ckpt: Path, language: str):
    """Fit BERTopic and pickle model + hard topic assignments.

    Probabilities are never computed (AppControl blocks _prediction_utils.pyd).
    """
    print(f"  Fitting BERTopic [lang={language}] (min_cluster_size={min_cs}, min_samples=5) …")

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cs,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=False,  # blocked by Windows AppControl
    )
    vectorizer_model = CountVectorizer(
        stop_words=list(stopwords(["vi", "en"])),
        min_df=2,
        ngram_range=(1, 2),
    )
    # PartOfSpeech removed: it overwrites topic_representations_ with only
    # noun/adjective tokens (~5 words), breaking the word cloud.
    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR":     MaximalMarginalRelevance(diversity=0.3),
    }
    topic_model = BERTopic(
        embedding_model=encoder,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ClassTfidfTransformer(),
        representation_model=representation_model,
        nr_topics="auto",
        min_topic_size=min_cs,
        top_n_words=100,
        calculate_probabilities=False,
        verbose=True,
    )
    topics, _ = topic_model.fit_transform(docs, embeddings)

    with open(ckpt, "wb") as f:
        pickle.dump({"model": topic_model, "topics": topics, "language": language}, f)
    print(f"  Checkpoint saved → {ckpt.name}")
    return topic_model, topics


def write_to_duckdb(run_id: str, topic_model, df: pd.DataFrame, topics: list, db_path: Path):
    """Write hard topic assignments and top-word labels to DuckDB.

    Uses INSERT OR REPLACE so re-runs with updated models overwrite stale rows.
    """
    con = duckdb.connect(str(db_path))

    topic_info = topic_model.get_topic_info()
    label_rows = []
    for _, row in topic_info.iterrows():
        tid = int(row["Topic"])
        kw_list = topic_model.get_topic(tid)
        top_words = ", ".join(w for w, _ in kw_list[:100]) if kw_list else ""
        label_rows.append((run_id, tid, top_words, int(row["Count"]), None, None))

    con.executemany(
        "INSERT OR REPLACE INTO TOPIC_LABELS "
        "(run_id, topic_id, top_words, n_docs, seed_topic, seed_score) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        label_rows,
    )

    review_rows = [
        (run_id, rid, int(tid), None)
        for rid, tid in zip(df["review_id"].tolist(), topics)
    ]
    con.executemany(
        "INSERT OR REPLACE INTO REVIEW_TOPICS (run_id, review_id, topic_id, prob) "
        "VALUES (?, ?, ?, ?)",
        review_rows,
    )

    con.close()
    n_topics = sum(1 for r in label_rows if r[1] != -1)
    print(f"  [{run_id}] DuckDB: {n_topics} topics, {len(review_rows):,} review rows")


def export_topic_csv(run_id: str, topic_model, df: pd.DataFrame, topics: list, out_dir: Path):
    """Export two CSVs:
    - <run_id>_topic_info.csv    : topic_id, n_docs, top_words, representative docs
    - <run_id>_review_topics.csv : review_id, hotel_id, review_year, language, topic_id
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    topic_info = topic_model.get_topic_info().copy()
    topic_info.insert(0, "run_id", run_id)
    topic_info.to_csv(out_dir / f"{run_id}_topic_info.csv", index=False, encoding="utf-8-sig")

    result = df[["review_id", "hotel_id", "review_year", "language", "distance2coastline"]].copy()
    result["topic_id"] = topics
    result.insert(0, "run_id", run_id)
    result.to_csv(out_dir / f"{run_id}_review_topics.csv", index=False, encoding="utf-8-sig")

    print(f"  [{run_id}] CSVs saved to {out_dir.name}/")


print("Helpers defined: fit_and_save | write_to_duckdb | export_topic_csv")

## Section 4 — Run: Distance-Band Models

Three BERTopic fits (beachfront / near-coast / inland).  
Each checkpoint is skipped if `checkpoints/<run_id>.pkl` already exists.

## Section 3b — Delete Stale Coast-Band Checkpoints

The existing `.pkl` files were fitted with the old default of `top_n_words=10`.
Run this cell **once** to remove them so Section 4 re-fits with `top_n_words=100`.

In [ ]:
# ── Delete stale pkl checkpoints ──────────────────────────────────────────
deleted_pkls = []
for run_id_base in BAND_SLICES:
    for lang in LANGUAGES:
        ckpt = CKPT_DIR / f"{run_id_base}_{lang}.pkl"
        if ckpt.exists():
            ckpt.unlink()
            deleted_pkls.append(ckpt.name)

# ── Delete stale DuckDB rows for all coast-band run_ids ───────────────────
# INSERT OR REPLACE in write_to_duckdb handles re-runs, but seed_topic/seed_score
# set by a later LLM step would be wiped by OR REPLACE. Explicit DELETE keeps
# the re-fit clean without touching year-slice rows.
coast_run_ids = [f"{base}_{lang}" for base in BAND_SLICES for lang in LANGUAGES]
placeholders  = ", ".join(f"'{r}'" for r in coast_run_ids)

con = duckdb.connect(str(DB_PATH))
con.execute(f"DELETE FROM TOPIC_LABELS  WHERE run_id IN ({placeholders})")
con.execute(f"DELETE FROM REVIEW_TOPICS WHERE run_id IN ({placeholders})")
con.close()

print(f"Deleted {len(deleted_pkls)} pkl(s): {deleted_pkls or 'none'}")
print(f"Cleared DuckDB rows for: {coast_run_ids}")

In [ ]:
for run_id_base, where_clause in BAND_SLICES.items():
    for lang in LANGUAGES:
        run_id = f"{run_id_base}_{lang}"
        ckpt   = CKPT_DIR / f"{run_id}.pkl"

        print(f"\n{'='*60}")
        print(f"Run    : {run_id}")
        print(f"Filter : {where_clause}  |  language={lang}")

        df, docs, embeddings = load_from_duckdb(
            db_path=DB_PATH,
            language=lang,
            extra_where=f"{where_clause} AND r.distance2coastline IS NOT NULL",
        )
        print(f"  Rows : {len(docs):,}")

        min_cs = SMALL_MIN_CLUSTER if len(docs) < SMALL_THRESHOLD else DEFAULT_MIN_CLUSTER

        if ckpt.exists():
            print(f"  Checkpoint found — loading {ckpt.name}")
            with open(ckpt, "rb") as f:
                saved = pickle.load(f)
            topic_model = saved["model"]
            topics      = saved["topics"]
        else:
            topic_model, topics = fit_and_save(run_id, docs, embeddings, min_cs, ckpt, lang)

        n_topics  = len(set(topics)) - (1 if -1 in topics else 0)
        n_outlier = sum(1 for t in topics if t == -1)
        print(f"  Topics: {n_topics}  |  Outliers: {n_outlier:,} ({n_outlier/len(topics)*100:.1f}%)")

        write_to_duckdb(run_id, topic_model, df, topics, DB_PATH)
        export_topic_csv(run_id, topic_model, df, topics, CSV_DIR)

print("\nDistance-band runs complete.")

## Section 5 — Run: Year-Slice Models (2018–2024)

Seven BERTopic fits, one per year. Years with < 10 k reviews automatically use a smaller `min_cluster_size`.

In [ ]:
for run_id_base, year in YEAR_SLICES.items():
    for lang in LANGUAGES:
        run_id = f"{run_id_base}_{lang}"
        ckpt   = CKPT_DIR / f"{run_id}.pkl"

        print(f"\n{'='*60}")
        print(f"Run    : {run_id}  (year={year}, language={lang})")

        df, docs, embeddings = load_from_duckdb(
            db_path=DB_PATH,
            language=lang,
            min_year=year,
            max_year=year,
        )
        print(f"  Rows : {len(docs):,}")

        min_cs = SMALL_MIN_CLUSTER if len(docs) < SMALL_THRESHOLD else DEFAULT_MIN_CLUSTER

        if ckpt.exists():
            print(f"  Checkpoint found — loading {ckpt.name}")
            with open(ckpt, "rb") as f:
                saved = pickle.load(f)
            topic_model = saved["model"]
            topics      = saved["topics"]
        else:
            topic_model, topics = fit_and_save(run_id, docs, embeddings, min_cs, ckpt, lang)

        n_topics  = len(set(topics)) - (1 if -1 in topics else 0)
        n_outlier = sum(1 for t in topics if t == -1)
        print(f"  Topics: {n_topics}  |  Outliers: {n_outlier:,} ({n_outlier/len(topics)*100:.1f}%)")

        write_to_duckdb(run_id, topic_model, df, topics, DB_PATH)
        export_topic_csv(run_id, topic_model, df, topics, CSV_DIR)

print("\nYear-slice runs complete.")

## Section 6 — Verify Checkpoint Inventory

In [8]:
all_run_ids = list(BAND_SLICES.keys()) + list(YEAR_SLICES.keys())

rows = []
for run_id in all_run_ids:
    ckpt = CKPT_DIR / f"{run_id}.pkl"
    size_mb = f"{ckpt.stat().st_size / 1e6:.1f} MB" if ckpt.exists() else "MISSING"
    rows.append({"run_id": run_id, "checkpoint": ckpt.name, "size": size_mb})

inventory = pd.DataFrame(rows)
print(inventory.to_string(index=False))

      run_id       checkpoint      size
coast_band_A coast_band_A.pkl  259.1 MB
coast_band_B coast_band_B.pkl  439.9 MB
coast_band_C coast_band_C.pkl 2049.8 MB
   year_2018    year_2018.pkl  171.3 MB
   year_2019    year_2019.pkl  216.0 MB
   year_2020    year_2020.pkl  165.0 MB
   year_2021    year_2021.pkl   52.1 MB
   year_2022    year_2022.pkl  182.7 MB
   year_2023    year_2023.pkl  365.7 MB
   year_2024    year_2024.pkl  243.5 MB


## Section 7 — Verify DuckDB Tables

In [9]:
con = duckdb.connect(str(DB_PATH), read_only=True)

print("=== TOPIC_LABELS — topics per run ===")
tl = con.execute("""
    SELECT run_id,
           COUNT(*) FILTER (WHERE topic_id != -1) AS n_topics,
           SUM(n_docs) FILTER (WHERE topic_id != -1) AS docs_in_topics,
           SUM(n_docs) FILTER (WHERE topic_id  = -1) AS outliers
    FROM TOPIC_LABELS
    GROUP BY run_id
    ORDER BY run_id
""").df()
print(tl.to_string(index=False))

print("\n=== REVIEW_TOPICS — assignments per run ===")
rt = con.execute("""
    SELECT run_id, COUNT(*) AS n_reviews
    FROM REVIEW_TOPICS
    GROUP BY run_id
    ORDER BY run_id
""").df()
print(rt.to_string(index=False))

con.close()

=== TOPIC_LABELS — topics per run ===
      run_id  n_topics  docs_in_topics  outliers
coast_band_A        27         11593.0   12135.0
coast_band_B        69         19974.0   20108.0
coast_band_C       261         77859.0  109659.0
   year_2018        11          8911.0    6755.0
   year_2019         7         10672.0    9090.0
   year_2020        22          8921.0    6151.0
   year_2021        37          3220.0    1522.0
   year_2022         9         16595.0     227.0
   year_2023        60         18043.0   15418.0
   year_2024        18         13252.0    8999.0

=== REVIEW_TOPICS — assignments per run ===
      run_id  n_reviews
coast_band_A      23728
coast_band_B      40082
coast_band_C     187518
   year_2018      15666
   year_2019      19762
   year_2020      15072
   year_2021       4742
   year_2022      16822
   year_2023      33461
   year_2024      22251


## Section 8 — Preview Top Words per Run

Quick sanity check — shows the 5 largest topics (excluding outlier -1) for each run.

In [10]:
con = duckdb.connect(str(DB_PATH), read_only=True)

preview = con.execute("""
    SELECT run_id, topic_id, n_docs, top_words
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY run_id ORDER BY n_docs DESC) AS rn
        FROM TOPIC_LABELS
        WHERE topic_id != -1
    ) t
    WHERE rn <= 5
    ORDER BY run_id, n_docs DESC
""").df()

con.close()

for run_id, grp in preview.groupby("run_id"):
    print(f"\n── {run_id} ──")
    for _, row in grp.iterrows():
        print(f"  topic {row.topic_id:>3}  ({row.n_docs:>6} docs)  {row.top_words}")


── coast_band_A ──
  topic   0  (  4594 docs)  staff, nice, beach, location, bin, friendly, clean, breakfast, helpful, pool
  topic   1  (  1376 docs)  sng, ngon, rt, nhnvin, khng, ti, khchsn, buffet, nhng, phng
  topic   2  (  1175 docs)  tuytvi, rt, tuytvi rt, tuytvi tuytvi, tuyt, lm, mi, excellent, rt tuytvi, qu tuytvi
  topic   3  (   943 docs)  staff, shower, bed, booked, bad, nice, water, floor, bathroom, reception
  topic   4  (   718 docs)  phng, khng, mi, khn, nhng, ko, ti, khchsn, phng khng, cng

── coast_band_B ──
  topic   0  (  4507 docs)  beach, bin, pool, nice, view, clean, staff, khchsn, schs, rt
  topic   1  (  3448 docs)  sng, ngon, rt, nhnvin, ti, khchsn, nhng, buffet, thnthin, sng ngon
  topic   2  (  1308 docs)  staff, booked, bed, shower, water, bathroom, bad, dirty, floor, reception
  topic   3  (   576 docs)  quay, ln, dp, nhittnh, quay ln, nhnvin, ti quay, schs, quay khchsn, thnthin
  topic   4  (   523 docs)  tin, gic, hpl, gitr, phhp, chtlng, phhp tin, gic h

## Section 9 — Checkpoint Viewer

Load any saved checkpoint and inspect its topics without re-fitting.

Set `RUN_ID` to any checkpoint name (without `.pkl`):

```
# Distance-band (after language-split re-run)
coast_band_A_en   coast_band_A_vi
coast_band_B_en   coast_band_B_vi
coast_band_C_en   coast_band_C_vi

# Year-slice (after language-split re-run)
year_2018_en   year_2018_vi   ...   year_2024_en   year_2024_vi
```

In [ ]:
# ── List all available checkpoints ────────────────────────────────────────
available = sorted(CKPT_DIR.glob("*.pkl"))
if available:
    print("Available checkpoints:")
    for p in available:
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.stem:<35}  {size_mb:>7.1f} MB")
else:
    print("No checkpoints found. Run Sections 4 & 5 first.")

In [ ]:
# ── Pick a checkpoint ─────────────────────────────────────────────────────
RUN_ID = "coast_band_A_en"   # ← change this to any run_id from the list above

ckpt = CKPT_DIR / f"{RUN_ID}.pkl"
if not ckpt.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt}")

with open(ckpt, "rb") as f:
    saved = pickle.load(f)

topic_model = saved["model"]
topics      = saved["topics"]
language    = saved.get("language", "unknown")

n_topics  = len(set(topics)) - (1 if -1 in topics else 0)
n_outlier = sum(1 for t in topics if t == -1)
n_total   = len(topics)

print(f"Run ID   : {RUN_ID}")
print(f"Language : {language}")
print(f"Topics   : {n_topics}")
print(f"Outliers : {n_outlier:,} / {n_total:,} ({n_outlier/n_total*100:.1f}%)")

In [ ]:
# ── Topic summary table (all topics, sorted by size) ──────────────────────
info = topic_model.get_topic_info()
info = info[info["Topic"] != -1].sort_values("Count", ascending=False).reset_index(drop=True)

# Build a clean display table with top-10 keywords
rows = []
for _, row in info.iterrows():
    tid     = int(row["Topic"])
    kw_list = topic_model.get_topic(tid)
    keywords = ", ".join(w for w, _ in kw_list[:10]) if kw_list else ""
    rows.append({"topic_id": tid, "n_docs": int(row["Count"]), "top_keywords": keywords})

summary = pd.DataFrame(rows)
print(f"[{RUN_ID}]  {len(summary)} topics\n")
print(summary.to_string(index=False))

In [ ]:
# ── Representative docs for a specific topic ──────────────────────────────
INSPECT_TOPIC = 0   # ← change to any topic_id from the table above
N_EXAMPLES    = 5

rep_docs = topic_model.get_representative_docs(INSPECT_TOPIC)
kw_list  = topic_model.get_topic(INSPECT_TOPIC)
keywords = ", ".join(w for w, _ in kw_list[:10]) if kw_list else ""

n_docs = int(info.loc[info["topic_id"] == INSPECT_TOPIC, "n_docs"].values[0]) if INSPECT_TOPIC in info["topic_id"].values else 0

print(f"Topic {INSPECT_TOPIC}  |  {n_docs:,} docs  |  lang={language}")
print(f"Keywords : {keywords}")
print("-" * 70)
for i, doc in enumerate(rep_docs[:N_EXAMPLES], 1):
    print(f"\n[{i}] {doc[:300]}")

## Section 10 — Diagnostics: Where Does the Garbling Come From?

Three checks in order:

1. **Representative docs** — are the raw input strings already garbled, or do they contain proper Vietnamese?
2. **CountVectorizer tokenization** — does the vectorizer strip diacritics or underscores?
3. **DuckDB round-trip** — are the stored `processed_text` strings clean?

Run the checkpoint loader (Section 9) first, then run these cells.

In [ ]:
# ── Check 1: are the representative docs themselves garbled? ──────────────
# get_representative_docs() returns the ORIGINAL doc strings passed to fit_transform.
# If these are garbled → the issue is in processed_text (DuckDB or preprocessing).
# If these are clean Vietnamese → the issue is in CountVectorizer tokenization.

DIAG_TOPIC = INSPECT_TOPIC  # reuses whatever topic you inspected above

rep_docs = topic_model.get_representative_docs(DIAG_TOPIC)

print(f"=== Representative docs for topic {DIAG_TOPIC} ===")
print(f"(These are the EXACT strings passed to fit_transform)\n")
for i, doc in enumerate(rep_docs, 1):
    print(f"[{i}] repr : {repr(doc[:120])}")
    print(f"     text : {doc[:120]}")
    print()

In [ ]:
# ── Check 2: what does the CountVectorizer actually tokenize? ─────────────
# Feed a known Vietnamese string through the same vectorizer that was used
# during fit_transform. Shows exactly which tokens are extracted.

import re

vi_sample = "khách_sạn sạch_sẽ nhân_viên thân_thiện giá_cả phù_hợp vị_trí dịch_vụ"

# Reproduce the token_pattern that was active when the checkpoint was fitted.
# Old checkpoints used r"(?u)\b\w\w+\b"; new vi checkpoints use r"[\w_][\w_]+".
vectorizer = topic_model.vectorizer_model
token_pattern = vectorizer.token_pattern
analyzer     = vectorizer.build_analyzer()

print(f"token_pattern : {token_pattern!r}")
print(f"ngram_range   : {vectorizer.ngram_range}")
print(f"strip_accents : {vectorizer.strip_accents!r}")
print()

tokens = analyzer(vi_sample)
print(f"Input  : {vi_sample!r}")
print(f"Tokens : {tokens}")
print()

# Also test each word individually
print("Per-word tokenization:")
for word in vi_sample.split():
    t = analyzer(word)
    print(f"  {word!r}  →  {t}")

In [ ]:
# ── Check 3: DuckDB round-trip for Vietnamese processed_text ──────────────
# Verifies the stored processed_text has proper diacritics before it even
# reaches BERTopic. Reads 5 vi rows and shows repr() so no terminal encoding
# issues can hide the actual byte content.

import duckdb

con = duckdb.connect(str(DB_PATH), read_only=True)
rows = con.execute("""
    SELECT r.review_id, p.processed_text
    FROM REVIEW_DATA r
    JOIN REVIEW_TEXT_PROCESSED p ON r.review_id = p.review_id
    WHERE r.language = 'vi'
      AND TRIM(COALESCE(p.processed_text, '')) != ''
    LIMIT 5
""").fetchall()
con.close()

print("=== DuckDB processed_text sample (vi) — repr avoids terminal encoding ===\n")
for rid, proc in rows:
    print(f"  review_id : {rid}")
    print(f"  processed : {repr(proc[:120])}")
    print()

# Conclusion guide:
# - If repr shows 'nhn_vin', 'kh_ch_s_n' etc → issue is in preprocess_to_duckdb.py
# - If repr shows 'nhân_viên', 'khách_sạn' etc → DuckDB is fine, issue is vectorizer

## Section 11 — Quick Vietnamese Sample Run (10 k reviews)

Fits one BERTopic model on a random 10,000-review Vietnamese sample.
Saves to `checkpoints/vi_sample_10k.pkl`.

**If `vi_sample_10k.pkl` already exists from before the language-fix, delete it first:**
```python
(CKPT_DIR / "vi_sample_10k.pkl").unlink(missing_ok=True)
```

Source of truth: all docs come from DuckDB (clean UTF-8), not from any CSV file.

In [ ]:
SAMPLE_SIZE = 10_000
RANDOM_SEED = 42
CKPT        = CKPT_DIR / "vi_sample_10k.pkl"

# ── Load all vi reviews ────────────────────────────────────────────────────
df_all, docs_all, emb_all = load_from_duckdb(db_path=DB_PATH, language="vi")
print(f"Total vi reviews available: {len(docs_all):,}")

# ── Random sample ──────────────────────────────────────────────────────────
import random
random.seed(RANDOM_SEED)
idx = sorted(random.sample(range(len(docs_all)), min(SAMPLE_SIZE, len(docs_all))))

docs       = [docs_all[i] for i in idx]
embeddings = emb_all[idx]
df         = df_all.iloc[idx].reset_index(drop=True)

print(f"Sampled        : {len(docs):,}")
print(f"Embedding shape: {embeddings.shape}")
print(f"\nSample doc [0] repr: {repr(docs[0][:120])}")
print(f"Sample doc [1] repr: {repr(docs[1][:120])}")

In [ ]:
# ── Fit BERTopic (or reload checkpoint) ───────────────────────────────────
if CKPT.exists():
    print(f"Checkpoint found — loading {CKPT.name}")
    with open(CKPT, "rb") as f:
        saved = pickle.load(f)
    vi_model  = saved["model"]
    vi_topics = saved["topics"]
else:
    vi_model = BERTopic(
        embedding_model=encoder,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42),
        hdbscan_model=HDBSCAN(
            min_cluster_size=20,
            min_samples=5,
            metric="euclidean",
            cluster_selection_method="eom",
            prediction_data=False,
        ),
        vectorizer_model=CountVectorizer(
            stop_words=list(stopwords(["vi", "en"])),
            min_df=2,
            ngram_range=(1, 2),
        ),
        ctfidf_model=ClassTfidfTransformer(),
        # PartOfSpeech removed: overwrites topic_representations_ with ~5 words
        representation_model={
            "KeyBERT": KeyBERTInspired(),
            "MMR":     MaximalMarginalRelevance(diversity=0.3),
        },
        nr_topics="auto",
        min_topic_size=20,
        top_n_words=100,
        calculate_probabilities=False,
        verbose=True,
    )
    vi_topics, _ = vi_model.fit_transform(docs, embeddings)

    with open(CKPT, "wb") as f:
        pickle.dump({"model": vi_model, "topics": vi_topics, "language": "vi"}, f)
    print(f"Checkpoint saved → {CKPT.name}")

n_topics  = len(set(vi_topics)) - (1 if -1 in vi_topics else 0)
n_outlier = sum(1 for t in vi_topics if t == -1)
print(f"\nTopics  : {n_topics}")
print(f"Outliers: {n_outlier:,} / {len(vi_topics):,} ({n_outlier/len(vi_topics)*100:.1f}%)")

In [ ]:
# ── Topic table + representative docs ─────────────────────────────────────
info = vi_model.get_topic_info()
info = info[info["Topic"] != -1].sort_values("Count", ascending=False).reset_index(drop=True)

print(f"{'topic_id':>8}  {'n_docs':>7}  top_keywords")
print("-" * 80)
for _, row in info.iterrows():
    tid      = int(row["Topic"])
    kw_list  = vi_model.get_topic(tid)
    keywords = ", ".join(w for w, _ in kw_list[:8]) if kw_list else ""
    print(f"{tid:>8}  {int(row['Count']):>7}  {keywords}")

print()

# Representative docs for the largest topic
top_tid   = int(info.iloc[0]["Topic"])
top_kws   = ", ".join(w for w, _ in vi_model.get_topic(top_tid)[:8])
rep_docs  = vi_model.get_representative_docs(top_tid)

print(f"=== Representative docs — topic {top_tid}: {top_kws} ===")
for i, doc in enumerate(rep_docs, 1):
    print(f"\n[{i}] {doc[:300]}")

## Section 12 — Re-fit Coast Bands: Cap at ≤ 25 Topics

Re-fits all six coast-band runs (`coast_band_A/B/C` × `en/vi`) with:

1. `min_cluster_size=10` — finer initial clusters
2. `reduce_outliers(strategy="c-tf-idf")` — reassign -1 docs to nearest topic
3. `update_topics()` — sync BERTopic internal state
4. `reduce_topics(nr_topics=25)` — merge down to ≤ 25 clean topics

Overwrites existing checkpoints and DuckDB rows for all coast bands.

In [ ]:
import math

TARGET_TOPICS = 25   # reduce_topics target — final count will be ≤ this

# min_samples = ceil(min_cluster_size / 2)
REFIT_MIN_CS = {
    "en": {"min_cluster_size": 5,  "min_samples": 3},   # ceil(5/2)  = 3
    "vi": {"min_cluster_size": 20, "min_samples": 10},  # ceil(20/2) = 10
}

# ── Wipe stale checkpoints & DuckDB rows ──────────────────────────────────
coast_run_ids = [f"{base}_{lang}" for base in BAND_SLICES for lang in LANGUAGES]

for run_id in coast_run_ids:
    ckpt = CKPT_DIR / f"{run_id}.pkl"
    if ckpt.exists():
        ckpt.unlink()
        print(f"Deleted checkpoint: {ckpt.name}")

placeholders = ", ".join(f"'{r}'" for r in coast_run_ids)
con = duckdb.connect(str(DB_PATH))
con.execute(f"DELETE FROM TOPIC_LABELS  WHERE run_id IN ({placeholders})")
con.execute(f"DELETE FROM REVIEW_TOPICS WHERE run_id IN ({placeholders})")
for tbl in ("TOPIC_ASPECTS", "TOPIC_LABELS"):
    try:
        con.execute(f"DELETE FROM {tbl} WHERE run_id IN ({placeholders})")
    except Exception:
        pass
con.close()
print(f"\nCleared DuckDB rows for: {coast_run_ids}")

# ── Re-fit loop ────────────────────────────────────────────────────────────
for run_id_base, where_clause in BAND_SLICES.items():
    for lang in LANGUAGES:
        run_id  = f"{run_id_base}_{lang}"
        ckpt    = CKPT_DIR / f"{run_id}.pkl"
        cs_cfg  = REFIT_MIN_CS[lang]

        print(f"\n{'='*60}")
        print(f"Run: {run_id}  |  min_cluster_size={cs_cfg['min_cluster_size']}  min_samples={cs_cfg['min_samples']}")

        df, docs, embeddings = load_from_duckdb(
            db_path=DB_PATH,
            language=lang,
            extra_where=f"{where_clause} AND r.distance2coastline IS NOT NULL",
        )
        print(f"  Docs: {len(docs):,}")

        # ── 1. Fit ────────────────────────────────────────────────────────
        umap_model = UMAP(
            n_neighbors=10,
            n_components=5,
            min_dist=0.0,
            metric="cosine",
            random_state=42,
        )
        hdbscan_model = HDBSCAN(
            min_cluster_size=cs_cfg["min_cluster_size"],
            min_samples=cs_cfg["min_samples"],
            metric="euclidean",
            cluster_selection_method="eom",
            prediction_data=False,
        )
        vectorizer_model = CountVectorizer(
            stop_words=list(stopwords(["vi", "en"])),
            min_df=2,
            ngram_range=(1, 2),
        )
        representation_model = {
            "KeyBERT": KeyBERTInspired(),
            "MMR":     MaximalMarginalRelevance(diversity=0.3),
        }
        topic_model = BERTopic(
            embedding_model=encoder,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            ctfidf_model=ClassTfidfTransformer(),
            representation_model=representation_model,
            nr_topics="auto",
            min_topic_size=cs_cfg["min_cluster_size"],
            top_n_words=100,
            calculate_probabilities=False,
            verbose=False,
        )
        topics, _ = topic_model.fit_transform(docs, embeddings)

        n_raw     = len(set(topics)) - (1 if -1 in topics else 0)
        n_outlier = sum(1 for t in topics if t == -1)
        print(f"  Raw topics: {n_raw}  |  Outliers: {n_outlier:,} ({n_outlier/len(topics)*100:.1f}%)")

        # ── 2. Reassign outliers ───────────────────────────────────────────
        new_topics = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf")
        topic_model.update_topics(docs, topics=new_topics)
        topics = new_topics

        n_after_ro = sum(1 for t in topics if t == -1)
        print(f"  After reduce_outliers: {n_after_ro} remaining outliers")

        # ── 3. Merge down to TARGET_TOPICS if needed ──────────────────────
        n_before = len(set(topics)) - (1 if -1 in topics else 0)
        if n_before > TARGET_TOPICS:
            topic_model.reduce_topics(docs, nr_topics=TARGET_TOPICS)
            topics = topic_model.topics_

        n_final   = len(set(topics)) - (1 if -1 in topics else 0)
        n_outlier = sum(1 for t in topics if t == -1)
        print(f"  FINAL: {n_final} topics  |  Outliers: {n_outlier:,} ({n_outlier/len(topics)*100:.1f}%)")

        # ── 4. Save ───────────────────────────────────────────────────────
        with open(ckpt, "wb") as f:
            pickle.dump({"model": topic_model, "topics": topics, "language": lang}, f)
        print(f"  Checkpoint saved → {ckpt.name}")

        write_to_duckdb(run_id, topic_model, df, topics, DB_PATH)
        export_topic_csv(run_id, topic_model, df, topics, CSV_DIR)

print("\n✓ All coast-band runs complete.")